# Model Load and Verification

This notebook loads the exported models from disk and verifies that they run inference correctly on a few samples.


**Section: Setup**

In [9]:
import json
from pathlib import Path
import numpy as np
import tensorflow as tf

DATA_DIR = Path('dataset')
MODELS_DIR = Path('models')
IMG_SIZE = 224


**Section: Load Class Names**

In [10]:
class_names_path = MODELS_DIR / 'class_names.json'
if not class_names_path.exists():
    raise FileNotFoundError(f"Missing {class_names_path}")

class_names = json.loads(class_names_path.read_text())
print('Classes:', class_names)


Classes: ['Cecidomyiidae', 'Chloropidae', 'Cicadellidae', 'Crambidae', 'Curculionidae', 'Delphacidae', 'Phlaeothripidae']


**Section: Sample Inputs**

In [11]:
# Collect a few sample images for testing
image_paths = []
for cls in class_names:
    cls_dir = DATA_DIR / cls
    if cls_dir.exists():
        for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
            image_paths.extend(cls_dir.glob(ext))

if not image_paths:
    raise FileNotFoundError('No images found in dataset/')

sample_paths = image_paths[:8]
print('Using sample images:', [str(p) for p in sample_paths])

def preprocess_for_model(path):
    img = tf.io.read_file(str(path))
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img

batch = tf.stack([preprocess_for_model(p) for p in sample_paths])


Using sample images: ['dataset/Cecidomyiidae/5 (618).jpg', 'dataset/Cecidomyiidae/5 (765).jpg', 'dataset/Cecidomyiidae/5 (762).jpg', 'dataset/Cecidomyiidae/5 (566).jpg', 'dataset/Cecidomyiidae/5 (495).jpg', 'dataset/Cecidomyiidae/5 (677).jpg', 'dataset/Cecidomyiidae/5 (551).jpg', 'dataset/Cecidomyiidae/5 (745).jpg']


**Section: Load Keras (.keras)**

In [12]:
keras_path = MODELS_DIR / 'pest_classifier.keras'
if keras_path.exists():
    keras_model = tf.keras.models.load_model(keras_path)
    preds = keras_model.predict(batch, verbose=0)
    print('Keras .keras loaded. Output shape:', preds.shape)
else:
    print('Keras .keras not found:', keras_path)


Keras .keras loaded. Output shape: (8, 7)


**Section: Load H5 (Full or Weights)**

In [13]:
h5_path = MODELS_DIR / 'pest_classifier.h5'
weights_path = MODELS_DIR / 'pest_classifier.weights.h5'

h5_loaded = False
if h5_path.exists():
    try:
        h5_model = tf.keras.models.load_model(h5_path)
        preds = h5_model.predict(batch, verbose=0)
        print('H5 full model loaded. Output shape:', preds.shape)
        h5_loaded = True
    except Exception as e:
        print('Failed to load full H5 model:', e)

if not h5_loaded and weights_path.exists():
    # Rebuild the model architecture to load weights
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.layers.Rescaling(1./127.5, offset=-1)(inputs)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(len(class_names), activation='softmax')(x)
    weights_model = tf.keras.Model(inputs, outputs)

    weights_model.load_weights(weights_path)
    preds = weights_model.predict(batch, verbose=0)
    print('Weights-only model loaded. Output shape:', preds.shape)


Failed to load full H5 model: No model config found in the file at models/pest_classifier.h5.
Weights-only model loaded. Output shape: (8, 7)


**Section: Load SavedModel**

In [14]:
saved_model_dir = MODELS_DIR / 'pest_classifier_savedmodel'
if saved_model_dir.exists():
    sm = tf.saved_model.load(str(saved_model_dir))
    infer = sm.signatures['serving_default']
    outputs = infer(tf.constant(batch))
    # Take first output tensor
    out = list(outputs.values())[0].numpy()
    print('SavedModel loaded. Output shape:', out.shape)
else:
    print('SavedModel not found:', saved_model_dir)


SavedModel loaded. Output shape: (8, 7)


**Section: Load TFLite**

In [15]:
tflite_path = MODELS_DIR / 'pest_classifier.tflite'
if tflite_path.exists():
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # TFLite expects a fixed input shape (often batch size 1)
    input_shape = input_details[0]['shape']
    batch_size = int(input_shape[0])

    # Use a single sample if model expects batch size 1
    input_data = batch.numpy()
    if batch_size == 1:
        input_data = input_data[:1]
    else:
        input_data = input_data[:batch_size]

    input_data = input_data.astype(input_details[0]['dtype'])
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])
    print('TFLite model loaded. Output shape:', output_data.shape)
else:
    print('TFLite not found:', tflite_path)


TFLite model loaded. Output shape: (1, 7)


2026-02-27 03:29:16.154724: E tensorflow/core/framework/node_def_util.cc:676] NodeDef mentions attribute use_inter_op_parallelism which is not in the op definition: Op<name=StatelessRandomGetKeyCounter; signature=seed:Tseed -> key:uint64, counter:uint64; attr=Tseed:type,default=DT_INT64,allowed=[DT_INT32, DT_INT64]> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node StatelessRandomGetKeyCounter}}
